Accompanying Code for: <br>
**Johannes Zeitler, Meinard Müller. "A Unified Perspective on CTC and SDTW using Differentiable DTW", submitted to IEEE Transactions on Audio, Speech, and Language Processing, 2025.**

Johannes Zeitler (johannes.zeitler@audiolabs-erlangen.de), 2025

### Description
For all files in the Schubert Winterreise Dataset (SWD), compute/save harmonic CQT and pitch class annotations

In [12]:
import os
import sys
import numpy as np, os, scipy, scipy.spatial, matplotlib.pyplot as plt, IPython.display as ipd
from numba import jit
import librosa
import libfmp.b, libfmp.c3, libfmp.c5
import pandas as pd, pickle, re
from numba import jit
from tqdm.notebook import tqdm
from libdl.data_preprocessing import compute_hopsize_cqt, compute_hcqt, compute_efficient_hcqt, compute_annotation_array_nooverlap

In [5]:
dataset_path = "path_to_SWD"
print(os.listdir(dataset_path))

['01_RawData', 'README.txt', '03_ExtraMaterial', '02_Annotations']


In [7]:
audio_folder = os.path.join(dataset_path, '01_RawData', 'audio_wav')

annotations_folder = os.path.join(dataset_path, '02_Annotations', 'ann_audio_note')

In [11]:
all_files = [f.split(".")[0] for f in os.listdir(audio_folder) if ".wav" in f]
all_files.sort()

In [10]:
hcqt_path_out = os.path.join("data", "hcqt")
ann_path_out = os.path.join("data", "pitchclass")

In [17]:
fs = 22050
bins_per_semitone = 3
num_octaves = 6
n_bins = bins_per_semitone*12*num_octaves
num_harmonics = 5
num_subharmonics = 1

In [ ]:
for file in tqdm(all_files):
    f_audio, fs_load = librosa.load(os.path.join(audio_folder, file+".wav"),
                                    sr=fs)
    f_hcqt, fs_hcqt, hopsize_cqt = compute_efficient_hcqt(f_audio, fs=22050, fmin=librosa.note_to_hz('C1'), fs_hcqt_target=50, \
                                                    bins_per_octave=bins_per_semitone*12, num_octaves=num_octaves, \
                                                    num_harmonics=num_harmonics, num_subharmonics=num_subharmonics)

    df = pd.read_csv(os.path.join(annotations_folder, file+".csv"), 
                     sep=';', skiprows=1, header=None)
    note_events = df.to_numpy()[:, :3]
            
    f_annot_pitchclass = compute_annotation_array_nooverlap(note_events.copy(), f_hcqt, fs_hcqt, annot_type='pitch_class', shorten=1.0)
    
    
    np.save(os.path.join(hcqt_path_out, file+'.npy'), f_hcqt)
    np.save(os.path.join(ann_path_out, file+'.npy'), f_annot_pitchclass)
